# 下準備

In [ ]:
!wget -N https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
!wget -N https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel

--2025-06-26 01:50:30--  https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28104 (27K) [text/plain]
Saving to: ‘deploy.prototxt’

deploy.prototxt     100%[===================>]  27.45K  --.-KB/s    in 0.003s  

Last-modified header missing -- time-stamps turned off.
2025-06-26 01:50:30 (8.24 MB/s) - ‘deploy.prototxt’ saved [28104/28104]

--2025-06-26 01:50:30--  https://raw.githubusercontent.com/opencv/opencv_3rdparty/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubus

In [ ]:
# ==== ライブラリ読み込み ====
from google.colab import drive
drive.mount('/content/drive')

from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from google.colab.output import eval_js, register_callback
from base64 import b64decode
from PIL import Image
from io import BytesIO
import base64
import IPython
import numpy as np
import cv2
import tensorflow as tf
import imutils

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==== AIモデル読み込み ====
print("[INFO] loading emotion model...")

# alexnet, vgg11, resnet18のどれかを選択
model_name = "alexnet"
model = tf.keras.models.load_model(f"drive/MyDrive/CNN_pic/model/{model_name}.keras")

[INFO] loading emotion model...


In [ ]:
# ==== 辞書 ====
number2label = {
    0: "angry", 1: "disgust", 2: "fear",
    3: "happy", 4: "sad", 5: "surprise", 6: "neutral"
}

emoji_paths = {
    0: "drive/MyDrive/CNN_pic/emoji/angry.png",
    1: "drive/MyDrive/CNN_pic/emoji/disgust.png",
    2: "drive/MyDrive/CNN_pic/emoji/fear.png",
    3: "drive/MyDrive/CNN_pic/emoji/happy.png",
    4: "drive/MyDrive/CNN_pic/emoji/sad.png",
    5: "drive/MyDrive/CNN_pic/emoji/surprise.png",
    6: "drive/MyDrive/CNN_pic/emoji/neutral.png"
}

# ==== 絵文字画像読み込み ====
emotion_emoji_images = {}
for idx, path in emoji_paths.items():
    emoji = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if emoji is not None and emoji.shape[2] == 4:
        emotion_emoji_images[idx] = emoji
    else:
        print(f"[WARN] 絵文字読み込み失敗 or RGBA形式でない: {path}")

In [ ]:
# ==== 顔検出器読み込み ====
print("[INFO] Loading face detection model...")
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

[INFO] Loading face detection model...


In [62]:
# ==== 顔検出＋絵文字貼付処理 ====
def face_detection_with_stamp(_img):
    _img = imutils.resize(_img, width=400)
    gray = cv2.cvtColor(_img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    for (x, y, w, h) in faces:
        face = _img[y:y+h, x:x+w]
        if face.size == 0:
            continue

        try:
            face_resized = cv2.resize(face, (224, 224))
            gray = cv2.cvtColor(face_resized, cv2.COLOR_BGR2GRAY)
            gray_3ch = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            face_tensor = tf.convert_to_tensor(gray_3ch, dtype=tf.float32) / 255.0
            result = model.predict(face_tensor[tf.newaxis], verbose=0)
            pred_id = int(np.argmax(result))
            label = number2label[pred_id]
        except Exception as e:
            print("[ERROR] 推論失敗:", e)
            continue

        if pred_id in emotion_emoji_images:
            emoji_img = emotion_emoji_images[pred_id]
            try:
                emoji_img = cv2.resize(emoji_img, (w, h))
                y1, y2 = y, y + emoji_img.shape[0]
                x1, x2 = x, x + emoji_img.shape[1]

                if y2 > _img.shape[0] or x2 > _img.shape[1]:
                    continue  # はみ出す場合スキップ

                alpha_s = emoji_img[:, :, 3] / 255.0
                alpha_l = 1.0 - alpha_s

                for c in range(3):
                    _img[y1:y2, x1:x2, c] = (alpha_s * emoji_img[:, :, c] +
                                             alpha_l * _img[y1:y2, x1:x2, c])

                # ===== テキスト表示（緑） =====
                text = f"{label} ({result[0][pred_id]*100:.1f}%)"
                font_scale = 0.5
                font_thickness = 1
                (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)
                text_x = x
                text_y = y - 5 if y - 5 > 10 else y + text_height + 5

                cv2.putText(_img, text, (text_x, text_y),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 0), font_thickness, cv2.LINE_AA)

            except Exception as e:
                print("[ERROR] 絵文字貼付失敗:", e)

    return _img

# ==== Colab カメラ → 推論処理 ====
def run(img_str):
    decimg = base64.b64decode(img_str.split(',')[1], validate=True)
    decimg = Image.open(BytesIO(decimg))
    decimg = np.array(decimg, dtype=np.uint8)
    decimg = cv2.cvtColor(decimg, cv2.COLOR_BGR2RGB)

    out_img = face_detection_with_stamp(decimg)

    _, encimg = cv2.imencode(".jpg", out_img, [int(cv2.IMWRITE_JPEG_QUALITY), 80])
    img_str = encimg.tobytes()
    img_str = "data:image/jpeg;base64," + base64.b64encode(img_str).decode('utf-8')
    return IPython.display.JSON({'img_str': img_str})

register_callback('notebook.run', run)

# ==== カメラ起動用JS ====
def use_cam(quality=0.8):
    js = Javascript('''
        async function useCam(quality) {
          const div = document.createElement('div');
          document.body.appendChild(div);

          const video = document.createElement('video');
          video.style.display = 'None';
          const stream = await navigator.mediaDevices.getUserMedia({video: true});
          div.appendChild(video);
          video.srcObject = stream;
          await video.play();

          display_size = 500
          const src_canvas = document.createElement('canvas');
          src_canvas.width  = display_size;
          src_canvas.height = display_size * video.videoHeight / video.videoWidth;
          const src_canvasCtx = src_canvas.getContext('2d');
          src_canvasCtx.translate(src_canvas.width, 0);
          src_canvasCtx.scale(-1, 1);
          div.appendChild(src_canvas);

          const dst_canvas = document.createElement('canvas');
          dst_canvas.width  = src_canvas.width;
          dst_canvas.height = src_canvas.height;
          const dst_canvasCtx = dst_canvas.getContext('2d');
          div.appendChild(dst_canvas);

          const btn_div = document.createElement('div');
          document.body.appendChild(btn_div);
          const exit_btn = document.createElement('button');
          exit_btn.textContent = 'Exit';
          var exit_flg = true
          exit_btn.onclick = function() {exit_flg = false};
          btn_div.appendChild(exit_btn);

          google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

          var send_num = 0
          _canvasUpdate();
          async function _canvasUpdate() {
                src_canvasCtx.drawImage(video, 0, 0, video.videoWidth, video.videoHeight, 0, 0, src_canvas.width, src_canvas.height);
                if (send_num<1){
                    send_num += 1
                    const img = src_canvas.toDataURL('image/jpeg', quality);
                    const result = google.colab.kernel.invokeFunction('notebook.run', [img], {});
                    result.then(function(value) {
                        parse = JSON.parse(JSON.stringify(value))["data"]
                        parse = JSON.parse(JSON.stringify(parse))["application/json"]
                        parse = JSON.parse(JSON.stringify(parse))["img_str"]
                        var image = new Image()
                        image.src = parse;
                        image.onload = function(){dst_canvasCtx.drawImage(image, 0, 0)}
                        send_num -= 1
                    })
                }
                if (exit_flg){
                    requestAnimationFrame(_canvasUpdate);
                }else{
                    stream.getVideoTracks()[0].stop();
                }
          };
        }
    ''')
    display(js)
    data = eval_js('useCam({})'.format(quality))

#  

In [61]:
# ==== 実行 ====
use_cam()

<IPython.core.display.Javascript object>